# 01 — Feature Extraction Pipeline
## Sensibilidad Temporal de Descriptores de Audio Artesanales

Este notebook implementa:
1. Configuración del entorno (Google Colab)
2. Carga y verificación de datasets (GTZAN, FMA-small, MagnaTagATune, IRMAS)
3. Extracción de features multi-escala (200 ms, 2 s, 5 s) — 7 descriptores × 3 escalas
4. Cache de features extraídos en Google Drive (**idempotente**: omite lo ya hecho)

**Salida:** Archivos `.npy` con features, labels, splits e índices por dataset en `FEATURES_ROOT`,
en el formato documentado en `README.md`.

**Ejecución:** Google Colab (GPU no necesaria; recomendada por RAM).

> Los features ya están extraídos en Drive. Este notebook solo re-extrae lo que falte,
> de modo que los `.npy` existentes siguen siendo válidos.

## 0. Setup & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q librosa scikit-learn tqdm soundfile

# Clone the repo to get the src/ modules
import os
REPO_URL = "https://github.com/Gabrieleeh32159/my_paper.git"
REPO_DIR = "/content/my_paper"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
import os
import sys
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
REPO_DIR = Path('/content/my_paper')

# Google Drive paths for data persistence
DRIVE_ROOT = Path('/content/drive/MyDrive/tsi_experiments')
DATA_ROOT = DRIVE_ROOT / 'data'
FEATURES_ROOT = DRIVE_ROOT / 'features'

# Create output directories
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)

# Add repo's experiments/ to path so `from src.X import ...` works
sys.path.insert(0, str(REPO_DIR / 'experiments'))

# Dataset paths (data lives on Drive)
DATASET_PATHS = {
    'gtzan': DATA_ROOT / 'gtzan',
    'fma_small': DATA_ROOT,  # FMA expects fma_small/ and fma_metadata/ under root
    'mtat': DATA_ROOT / 'magnatagatune',
    'irmas': DATA_ROOT / 'irmas',
}

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print(f"Repo source: {REPO_DIR}")
print(f"Drive root:  {DRIVE_ROOT}")
print(f"Data root:   {DATA_ROOT}")
print(f"Features:    {FEATURES_ROOT}")

In [ ]:
%cd {REPO_DIR}/experiments

from src.features import (
    FEATURE_DIMS, SCALES, TRACK_DIM,
    extract_multiscale_features, load_and_preprocess,
)
from src.data_loader import get_dataset

print("Modules loaded.")
print(f"Feature dimensions per frame: {sum(FEATURE_DIMS.values())} ({FEATURE_DIMS})")
print(f"Track vector dim per scale: {TRACK_DIM}")
print(f"Temporal scales (s): {SCALES}")

## 1. Data Availability

In [ ]:
# Verify available datasets
print("=== Dataset Availability ===")
for name, path in DATASET_PATHS.items():
    exists = path.exists()
    print(f"  {name:12s}: {'OK ' if exists else 'NO '} {path}")
    if exists:
        audio_exts = {'.mp3', '.wav', '.au', '.ogg'}
        n_files = sum(1 for f in path.rglob('*') if f.suffix in audio_exts)
        print(f"               {n_files} audio files found")

## 2. Multi-scale Feature Extraction (idempotent)

For each dataset we save (rows aligned by position):
`{ds}_short.npy`, `{ds}_medium.npy`, `{ds}_long.npy` (n×192, raw),
`{ds}_indices.npy`, `{ds}_labels.npy`, `{ds}_splits.npy`, `{ds}_errors.json`.

A dataset is **skipped** if its `{ds}_short.npy` already exists, so re-running is cheap and
the features already on Drive remain authoritative.

In [ ]:
from tqdm.auto import tqdm

SR = 16000

def feature_files(ds):
    return {
        'short':   FEATURES_ROOT / f"{ds}_short.npy",
        'medium':  FEATURES_ROOT / f"{ds}_medium.npy",
        'long':    FEATURES_ROOT / f"{ds}_long.npy",
        'indices': FEATURES_ROOT / f"{ds}_indices.npy",
        'labels':  FEATURES_ROOT / f"{ds}_labels.npy",
        'splits':  FEATURES_ROOT / f"{ds}_splits.npy",
        'errors':  FEATURES_ROOT / f"{ds}_errors.json",
    }

def already_extracted(ds):
    f = feature_files(ds)
    return all(f[k].exists() for k in ('short', 'medium', 'long', 'indices', 'labels', 'splits'))

def extract_dataset(ds, path, force=False):
    if already_extracted(ds) and not force:
        print(f"[{ds}] features already present -> SKIP (use force=True to re-extract)")
        return
    print(f"[{ds}] extracting from {path} ...")
    dataset = get_dataset(ds, str(path))
    n = len(dataset)
    short, medium, long_, indices, labels, splits, errors = [], [], [], [], [], [], []

    for i in tqdm(range(n), desc=f"{ds}"):
        fp = dataset.get_audio_path(i)
        try:
            y = load_and_preprocess(fp, sr=SR)
            if len(y) < SR:
                errors.append((i, str(fp), "too short"))
                continue
            feats = extract_multiscale_features(y, sr=SR)
            short.append(feats['short']); medium.append(feats['medium']); long_.append(feats['long'])
            indices.append(i)
            labels.append(dataset.get_label(i))
            splits.append(dataset.get_split(i))
        except Exception as e:
            errors.append((i, str(fp), str(e)))

    f = feature_files(ds)
    np.save(f['short'],   np.stack(short).astype(np.float32))
    np.save(f['medium'],  np.stack(medium).astype(np.float32))
    np.save(f['long'],    np.stack(long_).astype(np.float32))
    np.save(f['indices'], np.array(indices))
    np.save(f['labels'],  np.array(labels, dtype=object), allow_pickle=True)
    np.save(f['splits'],  np.array(splits, dtype=object), allow_pickle=True)
    with open(f['errors'], 'w') as fh:
        json.dump(errors, fh)
    print(f"[{ds}] done: {len(indices)} OK, {len(errors)} errors -> {FEATURES_ROOT}")

In [ ]:
# Run extraction for every available dataset (idempotent).
for ds, path in DATASET_PATHS.items():
    if not path.exists():
        print(f"[{ds}] data path missing -> skip")
        continue
    extract_dataset(ds, path, force=False)

### Optional: faster parallel extraction

For large datasets you can use the standalone, resumable worker
`src/extract_worker.py` (one fresh process per chunk, checkpoints every 50 tracks).
The sequential path above is sufficient and idempotent; the worker is only a speedup.

## 3. Verify saved features

In [ ]:
for ds in DATASET_PATHS:
    f = feature_files(ds)
    if not f['short'].exists():
        print(f"{ds}: not extracted"); continue
    X = np.load(f['short'])
    lab = np.load(f['labels'], allow_pickle=True)
    print(f"{ds}: short={X.shape} dtype={X.dtype}, labels={lab.shape}, "
          f"medium={np.load(f['medium']).shape}, long={np.load(f['long']).shape}")
    assert X.shape[1] == TRACK_DIM, "expected 192-d track vectors"
print("\nFeature format verified.")